#### Log Transform

In [ ]:
# Log Transform
df_m[features[0] + '_Log'] = np.log1p(df_m[features[0]])
# female
df_f[features[0] + '_Log'] = np.log1p(df_f[features[0]])

# remove
df_m = df_m.drop(columns=features)
df_f = df_f.drop(columns=features)

In [ ]:
plot_histogram(df_m, 'Level_SGOT_ALT_Log', txt='Male: Level SGOT ALT Log', bins=30)

In [ ]:
plot_histogram(df_f, 'AST_ALT_Ratio_Log', txt='Female: Level SGOT AST/ALT Log', bins=30)

### The "Shadow Variable" (Indicator) + Median Imputation

In [ ]:
# impute: median and use new feature for NANS
df_m['Level_SGOT_ALT_Log_IsMissing'] = df_m.Level_SGOT_ALT_Log.isna().astype(int)
df_m['Level_SGOT_AST_Log_IsMissing'] = df_m.Level_SGOT_AST_Log.isna().astype(int)
df_m['AST_ALT_Ratio_Log_IsMissing'] = df_m.AST_ALT_Ratio_Log.isna().astype(int)
# female
df_f['Level_SGOT_ALT_Log_IsMissing'] = df_f.Level_SGOT_ALT_Log.isna().astype(int)
df_f['Level_SGOT_AST_Log_IsMissing'] = df_f.Level_SGOT_AST_Log.isna().astype(int)
df_f['AST_ALT_Ratio_Log_IsMissing'] = df_f.AST_ALT_Ratio_Log.isna().astype(int)
# sanity check
(df_m['Level_SGOT_ALT_Log_IsMissing'].sum(), df_m['Level_SGOT_AST_Log_IsMissing'].sum(), df_m['AST_ALT_Ratio_Log_IsMissing'].sum(), 
 df_f['Level_SGOT_ALT_Log_IsMissing'].sum(), df_f['Level_SGOT_AST_Log_IsMissing'].sum(), df_f['AST_ALT_Ratio_Log_IsMissing'].sum()
)

In [ ]:
# impute
df_m['Level_SGOT_ALT_Log'] = df_m.Level_SGOT_ALT_Log.fillna(df_m.Level_SGOT_ALT_Log.median())
df_m['Level_SGOT_AST_Log'] = df_m.Level_SGOT_AST_Log.fillna(df_m.Level_SGOT_AST_Log.median())
df_m['AST_ALT_Ratio_Log'] = df_m.AST_ALT_Ratio_Log.fillna(df_m.AST_ALT_Ratio_Log.median())
# female
df_f['Level_SGOT_ALT_Log'] = df_f.Level_SGOT_ALT_Log.fillna(df_f.Level_SGOT_ALT_Log.median())
df_f['Level_SGOT_AST_Log'] = df_f.Level_SGOT_AST_Log.fillna(df_f.Level_SGOT_AST_Log.median())
df_f['AST_ALT_Ratio_Log'] = df_f.AST_ALT_Ratio_Log.fillna(df_f.AST_ALT_Ratio_Log.median())

In [ ]:
# info
features = u.get_feature_info(df, 'EducationLevel_CAN', cat=True)

In [ ]:
# Define the order
logical_order = sorted(df[feature[0]].unique())

# Create the "Type" definition
cat_type = CategoricalDtype(categories=logical_order, ordered=True)

# Apply it directly to the column
# Note: Any value NOT in logical_order (like "Unknown") will become NaN
df[feature[0]] = df[feature[0]].astype(cat_type)
#
feature = u.get_feature_info(df, feature[0], True)

In [ ]:
# mapping
mapping = {
    'NONE': 0,
    'GRADE SCHOOL (0-8)': 1,
    'HIGH SCHOOL (9-12) or GED': 2,
    'ATTENDED COLLEGE/TECHNICAL SCHOOL': 3,
    'ASSOCIATE/BACHELOR DEGREE': 4,
    'POST-COLLEGE GRADUATE DEGREE': 5,
    'Unknown': 99   # sentinel, not part of the scale
}

# new encoded feature
df[feature[0]] = df[feature[0]].map(mapping)

# Define the order
logical_order = sorted(mapping.values())

# Create the "Type" definition
cat_type = CategoricalDtype(categories=logical_order, ordered=True)

# Apply it directly to the column
# Note: Any value NOT in logical_order (like "Unknown") will become NaN
df[feature[0]] = df[feature[0]].astype(cat_type)
#
feature = u.get_feature_info(df, feature[0], True)

In [ ]:
# verify Missing is Random
u.check_informative_missingness(df, 'EducationLevel_CAN', target='TransplantSurvivalDay', unknown_val='Unknown')

In [ ]:
# mapping dictionary
collapse_map = results['mapping']
# update feature & mapping dataframe
df[feature[0]] = (df[feature[0]].map(collapse_map).astype("category"))
mapping_df.loc[mapping_df["feature"] == feature[0], "consolidate"] = True
# sanity check
df[feature[0]].isna().any()

In [ ]:
# values for NominalSurvivalRanker
class_values = {
    "r_thresh": 0.10,
    "min_n": 30,
    "method": "bonferroni",
    "suppress_pairwise": False,
    "rotation": 0
}
# NominalSurvivalRanker
mapping_df, results = ranker.cat_feature_ranker(data=df, mapping_data=mapping_df, 
                                                type_value=type_value, gender_value=gender_value,
                                                feature=feature, class_values=class_values, 
                                                re_run=False)

In [ ]:
# values for NominalSurvivalRanker
class_values = {"method": "fdr"}  # more relaxed
# NominalSurvivalRanker
mapping_df, results = ranker.cat_feature_ranker(data=df, mapping_data=mapping_df, 
                                                type_value=type_value, gender_value=gender_value,
                                                feature=feature, class_values=class_values, 
                                                re_run=True)

In [ ]:
# values for NominalSurvivalRanker
class_values = {"method": "fdr", "suppress_pairwise": True}  # more relaxed
# NominalSurvivalRanker
mapping_df, results = ranker.cat_feature_ranker(data=df, mapping_data=mapping_df, 
                                                type_value=type_value, gender_value=gender_value,
                                                feature=feature, class_values=class_values, 
                                                re_run=True)

In [ ]:
# mapping: collapse Missing & Unknown into Other: underpowered, small sample size
mapping = {'Unknown': 'Other',
           'Missing': 'Other'
          }
# map the change
df[feature[0]] = (df[feature[0]].map(mapping).fillna(df[feature[0]]))
# sanity check
ranker.mean_survival_by_category(df, feature[0])

In [ ]:
# values for NominalSurvivalRanker
class_values = {"method": "fdr"}
class_values = {"method": "bonferroni"}

In [ ]:
# sanity check
ranker.mean_survival_by_category(df, feature[0])

In [ ]:
# remove
mapping_df, remove_cols = ranker.update_remove_cols_and_mapping(mapping_data=mapping_df, remove_cols=remove_cols,
                                                                feature=feature, type_value=type_value, 
                                                                gender_value=gender_value)

##### **Note:** This feature does not have a statistically or practically significant impact on survival.

##### **Note:** Only keep this due to the difference in Other and will not collapse due to clinical interpretation.

##### **Note:** Do NOT collapse due to clinical interpretation.